# Lab 1: Observability & Evaluation

In this lab, we tackle the **Black Box Problem**: build a custom agent tracer, diagnose infinite loops, and implement circuit breakers. Then use **DeepEval** to evaluate performance.

## Setup
Install dependencies and load environment variables.

In [ ]:
# Run this cell first
%pip install -q litellm python-dotenv deepeval

import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv('OPENAI_API_KEY'):
    print('WARNING: OPENAI_API_KEY not found. Set it in your .env file.')
else:
    print('Environment loaded successfully.')

## Step 1: Instrumentation

Build an `AgentTracer` and inject it into a ReAct Agent.

In [ ]:
from dataclasses import dataclass, field
import json
import time
from typing import Optional

@dataclass
class ToolCallRecord:
    tool_name: str
    tool_input: dict
    tool_output: str
    duration_ms: float

@dataclass
class AgentStep:
    step_number: int
    reasoning: Optional[str] = None
    tool_calls: list = field(default_factory=list)
    cost_usd: float = 0.0

@dataclass
class Trace:
    trace_id: str
    agent_name: str
    steps: list = field(default_factory=list)
    status: str = 'running'
    total_cost_usd: float = 0.0

class AgentTracer:
    def __init__(self):
        self.traces = {}

    def start_trace(self, trace_id, agent_name):
        self.traces[trace_id] = Trace(trace_id=trace_id, agent_name=agent_name)
        return trace_id

    def log_step(self, trace_id, step):
        if trace_id in self.traces:
            self.traces[trace_id].steps.append(step)
            self.traces[trace_id].total_cost_usd += step.cost_usd

    def end_trace(self, trace_id, status='completed'):
        if trace_id in self.traces:
            self.traces[trace_id].status = status

    def print_trace(self, trace_id):
        t = self.traces.get(trace_id)
        if not t:
            print('Trace not found.')
            return
        print(f'\n=== TRACE: {t.trace_id} === Agent: {t.agent_name} | Status: {t.status}')
        for step in t.steps:
            print(f'  Step {step.step_number}: cost=${step.cost_usd:.4f}')
            for tc in step.tool_calls:
                print(f'    -> {tc.tool_name}({tc.tool_input}) [{tc.duration_ms:.0f}ms]')
        print(f'Total Cost: ${t.total_cost_usd:.4f}\n')

tracer = AgentTracer()
print('AgentTracer ready.')

Now define a broken tool that always fails, causing the agent to loop.

In [ ]:
import litellm
import uuid

def search(query):
    time.sleep(0.5)
    return 'Error: Database timeout. Please try again.'

def run_agent(query, tracer, max_steps=5):
    trace_id = str(uuid.uuid4())[:8]
    tracer.start_trace(trace_id, 'broken_react_agent')
    messages = [
        {'role': 'system', 'content': 'Use the search tool.'},
        {'role': 'user', 'content': query}
    ]
    tools = [{
        'type': 'function',
        'function': {
            'name': 'search',
            'description': 'Search the web.',
            'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}
        }
    }]
    for step_num in range(1, max_steps + 1):
        response = litellm.completion(model='gpt-4o-mini', messages=messages, tools=tools, temperature=0)
        msg = response.choices[0].message
        messages.append(msg.model_dump())
        try:
            cost = litellm.completion_cost(completion_response=response) or 0.0
        except Exception:
            cost = 0.0
        step_record = AgentStep(step_number=step_num, reasoning=msg.content, cost_usd=cost)
        if not msg.tool_calls:
            tracer.log_step(trace_id, step_record)
            tracer.end_trace(trace_id, 'completed')
            return msg.content, trace_id
        for tc in msg.tool_calls:
            if tc.function.name == 'search':
                args = json.loads(tc.function.arguments)
                t0 = time.time()
                result = search(args['query'])
                step_record.tool_calls.append(ToolCallRecord('search', args, result, (time.time()-t0)*1000))
                messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
        tracer.log_step(trace_id, step_record)
    tracer.end_trace(trace_id, 'max_steps_reached')
    return 'Reached max steps.', trace_id

answer, tid = run_agent('What is the capital of France?', tracer)
tracer.print_trace(tid)

## Step 2 & 3: Diagnosis and The Fix

Notice the cost accumulating and the same tool being called repeatedly. Now let's build a **Loop Detector** circuit breaker.

In [ ]:
class LoopDetector:
    def __init__(self, threshold=2):
        self.history = []
        self.threshold = threshold

    def check(self, name, args):
        call = (name, args)
        count = self.history.count(call)
        self.history.append(call)
        return count >= self.threshold

detector = LoopDetector()
print('Is loop?', detector.check('search', '{"query": "France"}'))  # False
print('Is loop?', detector.check('search', '{"query": "France"}'))  # False
print('Is loop?', detector.check('search', '{"query": "France"}'))  # True - triggers!

## Step 4: Verify with DeepEval

Use DeepEval's `TaskCompletionMetric` and `ToolCorrectnessMetric` to evaluate your agent.

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import TaskCompletionMetric, ToolCorrectnessMetric

test_cases = [
    LLMTestCase(
        input='What is the capital of France?',
        actual_output='Paris is the capital of France.',
        expected_output='Paris',
        tools_called=[ToolCall(name='search', input_parameters={'query': 'capital of France'})]
    ),
    LLMTestCase(
        input='What is 2 + 2?',
        actual_output='4',
        expected_output='4',
        tools_called=[]
    )
]

task_metric = TaskCompletionMetric(threshold=0.7)
tool_metric = ToolCorrectnessMetric(threshold=0.7)

print('Starting evaluation...')
results = evaluate(test_cases=test_cases, metrics=[task_metric, tool_metric], print_results=False)
print(f'Done. {sum(1 for r in results if r.success)}/{len(results)} passed.')

## Step 5: Replacing Manual Tracing with @observe

The `@observe` decorator replaces all the manual `AgentTracer` boilerplate. We also integrate the `LoopDetector` for a production-ready agent.

In [ ]:
from deepeval.tracing import observe
import litellm, json

class ObservableReactAgent:
    def __init__(self):
        self.detector = LoopDetector(threshold=1)

    @observe(type='tool')
    def execute_search(self, query):
        print(f'  [Tool] Searching: {query}')
        return 'Paris is the capital of France.'

    @observe(type='llm')
    def call_llm(self, messages, tools):
        return litellm.completion(model='gpt-4o-mini', messages=messages, tools=tools, temperature=0)

    @observe(type='agent')
    def run(self, query, max_turns=3):
        print(f'[Agent] Running: {query}')
        messages = [{'role': 'user', 'content': query}]
        tools = [{'type': 'function', 'function': {'name': 'execute_search', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}}}}]
        for _ in range(max_turns):
            response = self.call_llm(messages, tools)
            msg = response.choices[0].message
            messages.append(msg.model_dump())
            if not msg.tool_calls:
                return msg.content
            for tc in msg.tool_calls:
                if self.detector.check(tc.function.name, tc.function.arguments):
                    return 'Agent stopped: loop detected.'
                args = json.loads(tc.function.arguments)
                result = self.execute_search(args['query'])
                messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
        return 'Max turns reached.'

agent = ObservableReactAgent()
print('Answer:', agent.run('What is the capital of France?'))

### Why is @observe better?

1. **Zero Boilerplate**: No manual `time.time()` calls or record objects needed.
2. **Automatic Hierarchy**: Builds the Agent -> LLM -> Tool trace tree automatically.
3. **Integrated Safety**: Combine with `LoopDetector` for a production-ready agent.